# **RI Comparing Foundation Models Demo**



## 1. **Install Dependencies, Import Libraries, and Download Data**

In [ ]:
# Installing the dependencies
%pip install rime-sdk==2.6
%pip install python-dotenv

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from rime_sdk import Client
import os
from dotenv import load_dotenv, find_dotenv
from ri_public_examples.download_files import download_files

In [ ]:
download_files('generative/question_answering', 'question_answering') 

## 2. **Establish the RI Client**

To get started, provide the API credentials and the base domain/address of the RIME service. You can generate and copy an API token from the API Access Tokens Page under Workspace settings. For the domian/address of the RIME service, contact your admin. 

![img_1](https://drive.google.com/uc?id=1vMDhZii8yq22iuqSM8-Vqt3sZ2F3tPyz)

In [ ]:
## Load environment variables from dotenv file
load_dotenv(find_dotenv())
API_TOKEN = os.environ.get('API_TOKEN')
CLUSTER_URL = os.environ.get('CLUSTER_URL')
AGENT_ID = os.environ.get('AGENT_ID')
WORKSPACE_ID = os.environ.get('WORKSPACE_ID')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

In [ ]:
!cat /Users/alexderhacobian/Development/github/rime/.env

In [ ]:
client = Client(CLUSTER_URL, API_TOKEN)

## 3. **Create a New Project**

Below, create a project to store this and other future adversarial robustness stress test run results.

In [ ]:
description = (
    "Comparing GPT-3.5 and GPT-4 robustness "
    "and performance on Squad-V2 dataset"
)
project = client.create_project(
    name="Comparing Foundation Models Demo", 
    description=description,
    model_task="MODEL_TASK_QUESTION_ANSWERING"
)

## 4. **Create a New Integration with OpenAI** 

In [ ]:
integration_id = client.create_integration(
    workspace_id = WORKSPACE_ID,
    name = f"model_comparison_{datetime.now()}",
    integration_type = "INTEGRATION_TYPE_CUSTOM", 
    integration_schema = [
        {
            "name": "OPENAI_API_KEY",
            "sensitivity": "VARIABLE_SENSITIVITY_WORKSPACE_SECRET",
            "value": OPENAI_API_KEY, # FILL IN YOUR OPENAI API KEY HERE
        }
    ],
)

## 5. **Uploading the Model and Datasets**

##### 5.1. Uploading model files and registering GPT-3.5 model

In [ ]:
model_dir = client.upload_directory(
    Path('../../../python/rime/test_data/models/question_answering/'), 
    upload_path = "ri_public_examples_generative"
)

#Configure GPT-3.5 model
gpt35_model_path = model_dir + "/gpt35_model.py"

gpt35_model_id = project.register_model(
    name=f"gpt35_model_{datetime.now()}",
    model_config={
        "generative_language_model": {
            "model_path": gpt35_model_path,
            "system_prompt": (
                    "I am ChatGPT, a large language model trained by OpenAI,"
                    "based on the GPT-3.5 architecture.\nKnowledge cutoff: "
                    "2021-09\nCurrent date: {current_date}".format(current_date=datetime.today().strftime('%Y-%m-%d'))
            )
        }
    },    
    model_endpoint_integration_id = integration_id,
    agent_id = AGENT_ID,
    skip_validation = True
)

##### 5.1. Uploading model files and registering GPT-4 model

In [ ]:
#Configure GPT-4 model
gpt4_model_path = model_dir + "/gpt4_model.py"

gpt4_model_id = project.register_model(
    name=f"gpt4_model_{datetime.now()}",
    model_config={
        "generative_language_model": {
            "model_path": gpt4_model_path,
            "system_prompt": (
                    "I am ChatGPT, a large language model trained by OpenAI,"
                    "based on the GPT-4 architecture.\nKnowledge cutoff: "
                    "2021-09\nCurrent date: {current_date}".format(current_date=datetime.today().strftime('%Y-%m-%d'))
            )
        }
    },    
    model_endpoint_integration_id = integration_id,
    agent_id = AGENT_ID,
    skip_validation = True
)

##### 5.3. Uploading and registering a sample generative dataset

In [ ]:
eval_s3_path = client.upload_file(
    Path('question_answering/data/squad_v2_test_with_labels.json'), 
    upload_path = "ri_public_examples_generative"
)

eval_dataset_id = project.register_dataset(
    name=f"eval_dataset_{datetime.now()}",
    data_config={
        "connection_info": { 
            "data_file": {
                "path": eval_s3_path
            }
        },
        "data_params": {
            "label_col": "answer",
            "prompt_col": "prompt",
            "text_features": [
                "context",
                "question"]
        }
    }
)

## 6. **Configure Test Suite** 

In [ ]:
fact_sheet_path = client.upload_file(
    Path('question_answering/data/fact_sheet.txt'), 
    upload_path="ri_public_examples_generative"
)

tests_config = {
    "row_wise_consistency_with_knowledgebase": {
      "fact_sheet_path": fact_sheet_path,
      "run": True
    },
    "row_wise_pii_detection": {
      "run": False
    },
    "row_wise_toxicity": {
      "run": True
    },
    "row_wise_prompt_extraction_detection": {
      "run": True
    },
    "generative_char_substitution_attack": {
      "severity_thresholds": 0.85,
      "run": True,
    },
    "generative_lm_word_substitution_attack": {
      "run": False,
    }
}
test_suite_config = {"individual_tests_config": tests_config}

## 7. **Running a Stress Test**

##### 7.1. Writing GPT-3.5 stress testing configuration and running it

In [ ]:
gpt35_stress_test_config = {
    "run_name": "GPT-3.5 Adversarial Robustness",
    "data_info": {
        "eval_dataset_id": eval_dataset_id,
    },
    "model_id": gpt35_model_id,
    "categories": [
        "TEST_CATEGORY_TYPE_ADVERSARIAL",
        "TEST_CATEGORY_TYPE_BIAS_AND_FAIRNESS",
        "TEST_CATEGORY_TYPE_EVASION_ATTACK_DETECTION",
        "TEST_CATEGORY_TYPE_FACTUAL_AWARENESS",
        "TEST_CATEGORY_TYPE_MODEL_ALIGNMENT",
        "TEST_CATEGORY_TYPE_MODEL_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE",
        "TEST_CATEGORY_TYPE_TRANSFORMATIONS",
    ],
    "test_suite_config" : test_suite_config,
    "run_time_info": {
        "resource_request": {
            "ram_request_megabytes": "28000",
        },
        "random_seed" : "0"
    }
}
gpt35_stress_job = client.start_stress_test(
    gpt35_stress_test_config, project_id = project.project_id, agent_id = AGENT_ID
)
gpt35_stress_job.get_status(verbose=True, wait_until_finish=True)

##### 7.2. Writing GPT-4 stress testing configuration and running it

In [ ]:
gpt4_stress_test_config = {
    "run_name": "GPT-4 Adversarial Robustness",
    "data_info": {
        "eval_dataset_id": eval_dataset_id,
    },
    "model_id": gpt4_model_id,
    "categories": [
        "TEST_CATEGORY_TYPE_ADVERSARIAL",
        "TEST_CATEGORY_TYPE_BIAS_AND_FAIRNESS",
        "TEST_CATEGORY_TYPE_EVASION_ATTACK_DETECTION",
        "TEST_CATEGORY_TYPE_FACTUAL_AWARENESS",
        "TEST_CATEGORY_TYPE_MODEL_ALIGNMENT",
        "TEST_CATEGORY_TYPE_MODEL_PERFORMANCE",
        "TEST_CATEGORY_TYPE_SUBSET_PERFORMANCE",
        "TEST_CATEGORY_TYPE_TRANSFORMATIONS",
    ],
    "test_suite_config" : test_suite_config,
    "run_time_info": {
        "resource_request": {
            "ram_request_megabytes": "28000",
        }
    }
}
gpt4_stress_job = client.start_stress_test(
    gpt4_stress_test_config, project_id = project.project_id, agent_id = AGENT_ID
)
gpt4_stress_job.get_status(verbose=True, wait_until_finish=True)

## 8. **Analyzing and Querying Results**

Now that the test run is complete, we can check out the results in the RIME web interface.

In [ ]:
# GPT-3.5
gpt35_test_run = gpt35_stress_job.get_test_run()
gpt35_results_df = gpt35_test_run.get_result_df()
gpt35_results_df.head()

In [ ]:
# GPT-4
gpt4_test_run = gpt4_stress_job.get_test_run()
gpt4_results_df = gpt4_test_run.get_result_df()
gpt4_results_df.head()

In [ ]:
# Get the link to the GPT-3.5 stress test
print("https://"+ gpt35_test_run.get_link())

In [ ]:
# Get a link to the GPT-4 stress test
print("https://"+ gpt4_test_run.get_link())